# Vinculación determinística de documentos

In [47]:
import pandas as pd
import numpy as np
import os
os.chdir(r'C:\Users\Feder\Desktop\vinculacion_censo_rraa')

# la extracción y limpieza de las tablas está en "extracción y limpieza.ipynb"
rraa = pd.read_parquet(r'C:\Users\Feder\Desktop\vinculacion_censo_rraa\data\raw\rraa.parquet')
censo = pd.read_parquet(r'C:\Users\Feder\Desktop\vinculacion_censo_rraa\data\raw\censo.parquet')

In [56]:
documentos = censo[censo["id_pais_documento"] == "858"].groupby(['documento', 'documento_valido']).size().reset_index(name='count').sort_values(by = 'count', ascending=False)

documentos_genericos = documentos[(documentos["count"]>=10) | ((documentos["documento_valido"] == 0) & (documentos["count"] > 2))][["documento", "count"]]

## Análisis de los documentos

In [57]:
# armo cuadro con documentos genéricos válidos repetidos 10 o más veces
documentos_validos_10_veces = documentos[(documentos["count"]>=10) & (documentos["documento_valido"] == 1)].rename(columns={"count":"frecuencia"})[["documento", "frecuencia"]]

#documentos_validos_10_veces.to_csv(r'C:\Users\Feder\Desktop\vinculacion_censo_rraaa\output\documentos_validos_10_veces.csv')

print(documentos_validos_10_veces["frecuencia"].sum())

0


In [58]:
# armo cuadro con documentos genéricos no válidos repetidos 10 o más veces
documentos_novalidos_10_veces = documentos[(documentos["count"]>=10) & (documentos["documento_valido"] == 0)].rename(columns={"count":"frecuencia"})[["documento", "frecuencia"]]

print(documentos_novalidos_10_veces["frecuencia"].sum())

144


In [59]:
# armo cuadro con documentos genéricos válidos repetidos entre 3 y 10 veces
documentos_novalidos_3_veces = documentos[(documentos["count"]>2) & (documentos["count"]<10) & (documentos["documento_valido"] == 0)].rename(columns={"count":"frecuencia"})[["documento", "frecuencia"]]

print(documentos_novalidos_3_veces["frecuencia"].sum())

15


In [60]:
# check
22825+5167+458 == documentos_genericos["count"].sum()

np.False_

In [61]:
# cantidad de documentos extranjeros
len(censo[censo["id_pais_documento"] != "858"].dropna(subset=["id_pais_documento", "id_tipo_documento", "documento"]))

399

In [ ]:
# cantidad de personas sin documento
len(censo) - len(censo.dropna(subset = ["id_pais_documento", "id_tipo_documento", "documento"]))
print(len(censo) - len(censo.dropna(subset = ["id_pais_documento", "id_tipo_documento", "documento"]))()

SyntaxError: incomplete input (3214161266.py, line 3)

In [11]:
# documentos considerados que están repetidos
a = censo[(~censo["documento"].isin(documentos_genericos["documento"])) & (censo["id_pais_documento"] == "858")].dropna(subset=["id_pais_documento", "id_tipo_documento", "documento"]).duplicated(subset = "documento")

print(censo[(~censo["documento"].isin(documentos_genericos["documento"])) & (censo["id_pais_documento"] == "858")].dropna(subset=["id_pais_documento", "id_tipo_documento", "documento"])[a]["documento"].nunique())

del a

0


In [12]:
# cantidad de repeticiones
a = len(censo[(~censo["documento"].isin(documentos_genericos["documento"])) & (censo["id_pais_documento"] == "858")].dropna(subset=["id_pais_documento", "id_tipo_documento", "documento"]))

b = len(censo[(~censo["documento"].isin(documentos_genericos["documento"])) & (censo["id_pais_documento"] == "858")].dropna(subset=["id_pais_documento", "id_tipo_documento", "documento"]).drop_duplicates(subset="documento"))

a - b

0

In [13]:
# chequeo que sume 100%

2856997+234135+12572+6894+(22825+5167+458) == len(censo)

False

In [35]:
# elimino documentos genéricos, filas sin documento y me quedo con una fila por documento
censo2 = censo[(~censo["documento"].isin(documentos_genericos["documento"])) & (censo["id_pais_documento"] == "858")].dropna(subset=["id_pais_documento", "id_tipo_documento", "documento"]).drop_duplicates(subset=["id_pais_documento", "id_tipo_documento", "documento"])

print("se tienen", str(len(censo2)), "documentos distintos,", str(len(censo2[censo2["documento_valido"] == 1])), "son validos y", str(len(censo2[censo2["documento_valido"] == 0])), "no validos")

print(len(censo2))
print(len(censo))

se tienen 0 documentos distintos, 0 son validos y 0 no validos
0
1850


## Vinculación

In [33]:
result = pd.merge(censo2, rraa, on = ["id_pais_documento", "id_tipo_documento", "documento"], how = "outer")

vinculados = result[pd.notnull(result["id_censo"]) & pd.notnull(result["id_unico"])]
censo_no_vinculados = result[pd.notnull(result["id_censo"]) & pd.isnull(result["id_unico"])]
rraa_no_vinculados = result[pd.isnull(result["id_censo"]) & pd.notnull(result["id_unico"])]

print("vinculados:", len(vinculados))
print("censo no vinculados:", len(censo_no_vinculados))
print("rraa no vinculados:", len(rraa_no_vinculados))

vinculados: 0
censo no vinculados: 0
rraa no vinculados: 2150


## Censo no vinculados

In [16]:
len(censo_no_vinculados)

0

In [17]:
censo_no_vinculados.value_counts("documento_valido_x", dropna = False)

Series([], Name: count, dtype: int64)

In [18]:

documentos_no_vinculados_validos = censo_no_vinculados[censo_no_vinculados["documento_valido_x"] == 1][["id_pais_documento", "id_tipo_documento", "documento"]]

#documentos_no_vinculados_validos.to_csv(r'C:\Users\lpescetto\Desktop\vinculacion_censo_rraaa\output\documentos_no_vinculados_validos.csv')

In [19]:
censo_no_vinculados.value_counts(["tipo_caso", "documento_valido_x"], dropna=False)

Series([], Name: count, dtype: int64)

## Vinculados

In [20]:
len(vinculados)

0

In [21]:
vinculados.value_counts("documento_valido_x")

Series([], Name: count, dtype: int64)

In [23]:
# from load_data.rraa import get_id_estado

# # me traigo la etiqueta de id_estado
# id_estado = get_id_estado()

In [24]:
vinculados_id_estado = pd.merge(vinculados, id_estado, left_on = "id_unico", right_on = "id_estadistico_persona", how = "left")

vinculados_id_estado.value_counts("id_estado_y", dropna=False).sort_values(ascending=False)

NameError: name 'id_estado' is not defined

## Asignación de id_censo a cada documento vinculado

Cuando el documento está duplicado, le asigno el id_censo que tiene la fecha de nacimiento más cercana a la de rraa.

In [24]:
from load_data.censo import get_personas_ampliada

departamento = get_personas_ampliada(["departamento_r"])

In [25]:
vinculados_con_dups = pd.merge(vinculados.rename(columns={"fecha_nacimiento_y": "fecha_nacimiento"})[["id_pais_documento", "id_tipo_documento", "documento", "id_unico", "fecha_nacimiento"]], censo.rename(columns={"fecha_nacimiento": "fecha_nacimiento_censo"}),
                               how = "left")

vinculados_con_dups_depto = pd.merge(vinculados_con_dups, departamento, how = "left")

vinculados_dups = vinculados_con_dups_depto.groupby("id_unico").filter(lambda group: len(group)>1)

In [26]:
print("documentos vinculados duplicados:", len(vinculados_dups.drop_duplicates(subset = ["id_unico"])))

print("documentos vinculados no duplicados:", len(vinculados_con_dups[["id_unico", "id_censo"]].drop_duplicates(subset = ["id_unico"], keep=False)))

vinculados_dups["comp_fecha"] = abs((vinculados_dups["fecha_nacimiento"] - vinculados_dups["fecha_nacimiento_censo"]).dt.days).fillna(0)

min_fecha = vinculados_dups.groupby('id_unico')['comp_fecha'].transform("min")

vinculados4 = vinculados_dups[vinculados_dups["comp_fecha"] == min_fecha]

#vinculados4 = vinculados4.groupby("id_unico").filter(lambda group: group["departamento_r"].nunique() > 1)
# tenemos 106 documentos-fecnac duplicados en distintos departamentos :/

vinculados_dedup = vinculados4.drop_duplicates(subset = "id_unico")

documentos vinculados duplicados: 12260
documentos vinculados no duplicados: 2833084


In [27]:
# check
2833084+12260 == len(vinculados)

True

In [28]:
vinculados_con_id_censo = pd.concat(
    [vinculados_con_dups[["id_unico", "id_censo"]].drop_duplicates(subset = ["id_unico"], keep=False),
    vinculados_dedup[["id_unico", "id_censo"]]],
    axis = 0, ignore_index=True)


In [29]:
# check
print(len(vinculados_con_id_censo),
len(vinculados_con_id_censo.drop_duplicates(subset="id_unico")),
len(vinculados_con_id_censo.drop_duplicates(subset="id_censo")))

2845344 2845344 2845344


In [30]:
from utils.postgresFunctions import crear_tabla_en_batches

id_estadistico_vinculados = vinculados_con_id_censo.rename(columns={"id_unico": "id_estadistico_persona"})

crear_tabla_en_batches(id_estadistico_vinculados, "registro_poblacion", "vinculados_censo_documento")

Creando tabla registro_poblacion.vinculados_censo_documento en DW
Progreso: 0.00%
Progreso: 33.33%
Progreso: 66.67%
La tabla registro_poblacion.vinculados_censo_documento fue actualizada.
